<a href="https://colab.research.google.com/github/ricardoproenca/AI/blob/main/AnaliseSentimento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalar dependência
!pip install pymupdf --break-system-packages

In [ ]:
import torch

# Verificar se GPU está disponível
print(f"GPU disponivel : {torch.cuda.is_available()}")
device = print(f"Device         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
device = 0 if torch.cuda.is_available() else -1  # 0 = GPU, -1 = CPU
print(f"Usando device : {'GPU' if device == 0 else 'CPU'}")

In [ ]:
t0 = time.time()
import fitz  # pymupdf
import time
from datetime import timedelta
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
inicio_total = time.time()

In [ ]:
# ── 1. Carregar o modelo ──────────────────────────────────────────
#model_name = "ProsusAI/finbert"
model_name = "lucas-leme/FinBERT-PT-BR"
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=False)
model = AutoModelForSequenceClassification.from_pretrained(model_name, local_files_only=False)

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

In [ ]:
print(f"[1/5] Carregar o modelo...              {time.time()-t0:.2f}s")
t1 = time.time()-t0

In [ ]:
t0 = time.time()
from google.colab import drive

# Monta o Google Drive
drive.mount('/content/drive')

In [ ]:
import os
t0 = time.time()
# Liste os arquivos para encontrar o caminho correto
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.pdf'):
            print(os.path.join(root, file))

In [ ]:
print(f"[2/5] Montar Drive...              {time.time()-t0:.2f}s")
t2 = time.time()-t0

In [ ]:
# ── 2. Ler o PDF ─────────────────────────────────────────────────
t0 = time.time()
import fitz

# Substitua pelo caminho real encontrado acima
#caminho_pdf = "/content/drive/MyDrive/Formulário de Referência 2025_2026 - Raízen S.A..pdf"
caminho_pdf = "/content/drive/MyDrive/FR Bradespar_PT_V7 2025.pdf"

def ler_pdf(caminho):
    doc = fitz.open(caminho)
    texto = ""
    for pagina in doc:
        texto += pagina.get_text()
    return texto.strip()

texto_pdf = ler_pdf(caminho_pdf)
print(f"Total de caracteres: {len(texto_pdf)}")
print(texto_pdf[:500])  # Mostra os primeiros 500 caracteres

In [ ]:
print(f"[3/5] Ler o PDF...              {time.time()-t0:.2f}s")
t3 = time.time()-t0

In [ ]:
# ── 3. Classificar ───────────────────────────────────────────────
# O FinBERT tem limite de 512 tokens — dividir em parágrafos
t0 = time.time()
paragrafos = [p.strip() for p in texto_pdf.split("\n") if len(p.strip()) > 50]

for paragrafo in paragrafos:
    resultado = classifier(paragrafo, truncation=True, max_length=512)
    print(f"[{resultado[0]['label']}] ({resultado[0]['score']:.4f}) {paragrafo[:80]}...")

In [ ]:
print(f"[4/5] Classificar...              {time.time()-t0:.2f}s")
t4 = time.time()-t0

In [ ]:
# ── 4. Totalizar ───────────────────────────────────────────────
t0 = time.time()
from collections import Counter

labels = []
for paragrafo in paragrafos:
    resultado = classifier(paragrafo, truncation=True, max_length=512)
    labels.append(resultado[0]["label"])

contagem = Counter(labels)
total = len(labels)

print("=" * 40)
print("RESULTADO DA ANALISE DE SENTIMENTO")
print("=" * 40)
for label, count in sorted(contagem.items()):
    pct = count / total * 100
    bar = "#" * int(pct / 2)  # usa # no lugar de █
    print(f"{label:10s}: {count:4d} ({pct:5.1f}%) {bar}")
print("-" * 40)
print(f"{'TOTAL':10s}: {total:4d}")

In [ ]:
print(f"[5/5] Totalizar...              {time.time()-t0:.2f}s")
t5 = time.time()-t0

In [ ]:
tempo_total = time.time() - inicio_total

In [ ]:
t0 = time.time()
import fitz  # pymupdf
import numpy as np
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')

def descrever_pdf(caminho_pdf):
    doc = fitz.open(caminho_pdf)

    # ── Métricas por página ──────────────────────────────────────
    paginas_info = []
    for i, pagina in enumerate(doc):
        texto_pag  = pagina.get_text()
        palavras   = texto_pag.split()
        frases_pag = sent_tokenize(texto_pag, language="portuguese")

        paginas_info.append({
            "pagina"    : i + 1,
            "caracteres": len(texto_pag),
            "palavras"  : len(palavras),
            "frases"    : len(frases_pag),
            "linhas"    : len(texto_pag.split("\n")),
        })

    # ── Totais ───────────────────────────────────────────────────
    total_chars    = sum(p["caracteres"] for p in paginas_info)
    total_palavras = sum(p["palavras"]   for p in paginas_info)
    total_frases   = sum(p["frases"]     for p in paginas_info)
    total_linhas   = sum(p["linhas"]     for p in paginas_info)
    chars_por_pag  = [p["caracteres"]    for p in paginas_info]
    total_pages    = len(doc) # Get total pages here

    # ── Tokens (FinBERT) ─────────────────────────────────────────
    total_tokens = len(tokenizer(texto_pdf, truncation=False)["input_ids"])

    # ── Exibir ───────────────────────────────────────────────────
    print("=" * 50)
    print(" DESCRICAO DO PDF")
    print("=" * 50)
    print(f"  Arquivo           : {caminho_pdf.split('/')[-1]}")
    print(f"  Paginas           : {total_pages}")
    print(f"  Caracteres total  : {total_chars:,}")
    print(f"  Palavras total    : {total_palavras:,}")
    print(f"  Frases total      : {total_frases:,}")
    print(f"  Linhas total      : {total_linhas:,}")
    print(f"  Tokens (FinBERT)  : {total_tokens:,}")
    print(f"\n  Caracteres por pagina:")
    print(f"    Media           : {np.mean(chars_por_pag):,.0f}")
    print(f"    Minimo          : {min(chars_por_pag):,} (pag. {np.argmin(chars_por_pag)+1})")
    print(f"    Maximo          : {max(chars_por_pag):,} (pag. {np.argmax(chars_por_pag)+1})")
    print(f"\n  Paginas vazias    : {sum(1 for p in paginas_info if p['caracteres'] < 50)}")

    print(f"\n{'=' * 50}")
    print(f" DETALHE POR PAGINA")
    print(f"{'=' * 50}")
    print(f"  {'Pag':>4} | {'Chars':>8} | {'Palavras':>8} | {'Frases':>7} | {'Linhas':>7}")
    print(f"  {'-'*4}-+-{'-'*8}-+-{'-'*8}-+-{'-'*7}-+-{'-'*7}")
    for p in paginas_info:
        print(f"  {p['pagina']:>4} | {p['caracteres']:>8,} | {p['palavras']:>8,} | {p['frases']:>7,} | {p['linhas']:>7,}")

    return paginas_info, total_chars, total_pages

# Executar
paginas_info, total_chars_final, total_pages_final = descrever_pdf(caminho_pdf)

In [ ]:
# ── Totalizador ──────────────────────────────────────────────
print(f"\n{'=' * 50}")
print(f" TOTALIZADOR DE EXECUCAO")
print(f"{'=' * 50}")
print(f"  [1/5] Carregar modelo : {t1:6.2f}s")
print(f"  [2/5] Montar Drive    : {t2:6.2f}s")
print(f"  [3/5] Ler o PDF       : {t3:6.2f}s")
print(f"  [4/5] Classificar     : {t4:6.2f}s")
print(f"  [5/5] Totalizar       : {t5:6.2f}s")
print(f"  Device                : {'GPU ' + torch.cuda.get_device_name(0) if device == 0 else 'CPU'}")
print(f"  {'─' * 36}")
print(f"  Tempo total           : {str(timedelta(seconds=round(tempo_total)))}")
print(f"  Velocidade            : {total_pages_final/tempo_total:.1f} frases/seg")
print(f"  Chars/segundo         : {total_chars_final/tempo_total:,.0f}")
print(f"{'=' * 50}")